# read_osraf2night.ipynb

Dynamic spectrum reader for the Tremdorf OSRA data — **two nightly windows**:

| Window | Time range (UT) | Description |
|--------|----------------|-------------|
| 1 | 03:00 → 07:00 | Night / Dawn |
| 2 | 15:00 → 20:00 | Afternoon / Evening |

Each window is plotted hour-by-hour as a resampled dynamic spectrum (frequency vs time).

## 1  Imports, file reading & normalisation

In [ ]:
import matplotlib.dates as mdates
import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
import scipy.ndimage
import detectRadioburst_OSRA as drb

# ── File and frequency initialisation ─────────────────────────────────────────
fname = "/net/lyot/scratch3/vocks/OSRA/2003/CD_300/031027_300.roh"

# Define the frequency array for f2 (400 MHz → 200 MHz, 256 channels)
f2     = 400.0 - np.array(range(256)) * 200.0 / 255.0
f_fits = f2

# ── Determine number of records from file size ────────────────────────────────
file_stats = os.stat(fname)
a1 = int(file_stats.st_size / 1040 + 0.5)

# Placeholder time array (will be overwritten record-by-record below)
dummy_start = np.datetime64('2025-02-01T15:23:15.00')
t_fits = dummy_start + np.linspace(0, 1, a1).astype('timedelta64[D]')

# Spectrum array for the f2 frequency range
dyspec = np.zeros((a1, 256), dtype=np.ubyte)

# ── Read binary records ───────────────────────────────────────────────────────
file = open(fname, "rb")
for i in range(a1):
    data_chunk    = file.read(1040)
    np_data_chunk = np.frombuffer(data_chunk, dtype=np.uint8)

    year = int(np_data_chunk[0] / 16) * 10 + (np_data_chunk[0] & 15)
    year = year + 1900 if year > 50 else year + 2000

    month       = int(np_data_chunk[1] / 16) * 10 + (np_data_chunk[1] & 15)
    day         = int(np_data_chunk[2] / 16) * 10 + (np_data_chunk[2] & 15)
    hour        = int(np_data_chunk[3] / 16) * 10 + (np_data_chunk[3] & 15)
    minute      = int(np_data_chunk[4] / 16) * 10 + (np_data_chunk[4] & 15)
    second      = int(np_data_chunk[5] / 16) * 10 + (np_data_chunk[5] & 15)
    microsecond = 100000 * np_data_chunk[6]

    t_fits[i]   = datetime.datetime(year, month, day, hour, minute, second, microsecond)
    dyspec[i,:] = np_data_chunk[272:528]   # f2 frequency slice
file.close()

# ── Per-channel normalisation ─────────────────────────────────────────────────
# Guard against flat channels (Imax == Imin) which would cause divide-by-zero.
dyspec_norm = np.zeros((a1, 256), dtype=np.ubyte)
for j in range(256):
    Imax = np.amax(dyspec[:, j])
    Imin = np.amin(dyspec[:, j])
    if Imax > Imin:                                          # normal channel
        dyspec_norm[:, j] = (dyspec[:, j] - Imin) / (Imax - Imin) * 255
    # else: channel is flat → leave as zeros (no signal / dead channel)
dyspec_norm = dyspec_norm.T   # transpose for imshow

print(f"Loaded {a1} records. Time range: {t_fits[0]} → {t_fits[-1]}")


## 2  Plotting helper

In [ ]:
def plot_window(start_time, end_time, label=""):
    """
    Iterate hour-by-hour over [start_time, end_time) and plot each hourly
    dynamic spectrum panel.  Both arguments must be np.datetime64 strings.
    """
    current_time = np.datetime64(start_time)
    end          = np.datetime64(end_time)

    while current_time < end:
        next_time = current_time + np.timedelta64(1, 'h')

        time_mask      = (t_fits >= current_time) & (t_fits < next_time)
        t_fits_cutout  = t_fits[time_mask]
        dyspec_cutout  = dyspec[time_mask, :]
        a2             = len(t_fits_cutout)

        if a2 == 0:
            print(f"  No data in {current_time} – {next_time}, skipping.")
            current_time = next_time
            continue

        # Resample to target_steps along the time axis
        target_steps  = 5000
        new_dyspec    = np.zeros((target_steps, len(f_fits)), dtype=np.uint8)
        zoomed_array  = scipy.ndimage.zoom(dyspec_cutout, (target_steps / a2, 1))
        new_dyspec[:] = zoomed_array

        # Normalise resampled data
        new_dyspec_norm = (
            (new_dyspec - new_dyspec.min(axis=0)) /
            (new_dyspec.ptp(axis=0) + 1e-6) * 255
        ).astype(np.uint8)

        # Convert time stamps for matplotlib
        t_plot_cutout = mdates.date2num(t_fits_cutout)

        # ── Plot ──────────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(12, 6))
        plot_array = np.flip(new_dyspec_norm.T, axis=0)
        im = ax.imshow(
            plot_array, aspect='auto', cmap='plasma',
            extent=[t_plot_cutout[0], t_plot_cutout[-1], f_fits[0], f_fits[-1]]
        )
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_xlabel('Time (UT)')
        ax.set_ylabel('Frequency (MHz)')
        window_tag = f"{current_time.astype('datetime64[h]')} – {next_time.astype('datetime64[h]')}"
        ax.set_title(f'Dynamic Spectrum  {window_tag}  {label}', fontweight='bold')
        plt.colorbar(im, ax=ax, label='Intensity')
        ax.set_yticks(np.linspace(f_fits[0], f_fits[-1], 10))
        plt.tight_layout()
        plt.show()

        current_time = next_time


## 3  Window 1 — Night / Dawn: 03:00 → 07:00 UT

In [ ]:
# ── Window 1 : Night / Dawn  03:00 UT → 07:00 UT ─────────────────────────────
print("=" * 60)
print("Plotting Window 1 : 03:00 – 07:00 UT")
print("=" * 60)

plot_window(
    start_time = '2003-10-27T03:00:00.000',
    end_time   = '2003-10-27T07:00:00.000',
    label      = '[Night / Dawn]'
)


## 4  Window 2 — Afternoon / Evening: 15:00 → 20:00 UT

In [ ]:
# ── Window 2 : Afternoon / Evening  15:00 UT → 20:00 UT ──────────────────────
print("=" * 60)
print("Plotting Window 2 : 15:00 – 20:00 UT")
print("=" * 60)

plot_window(
    start_time = '2003-10-27T15:00:00.000',
    end_time   = '2003-10-27T20:00:00.000',
    label      = '[Afternoon / Evening]'
)
